In [69]:
import urllib.request
import pandas as pd
import numpy as np

# Download the UCI Parkinson's Dataset (tabular measurements of voice features)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data'
data_path = 'parkinsons.csv'
urllib.request.urlretrieve(url, data_path)

# Load and describe the dataset
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
print(df.head())
print(df.info())
print(df['status'].value_counts())

Dataset Shape: (195, 24)
             name  MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  \
0  phon_R01_S01_1      119.992       157.302        74.997         0.00784   
1  phon_R01_S01_2      122.400       148.650       113.819         0.00968   
2  phon_R01_S01_3      116.682       131.111       111.555         0.01050   
3  phon_R01_S01_4      116.676       137.871       111.366         0.00997   
4  phon_R01_S01_5      116.014       141.781       110.655         0.01284   

   MDVP:Jitter(Abs)  MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  ...  \
0           0.00007   0.00370   0.00554     0.01109       0.04374  ...   
1           0.00008   0.00465   0.00696     0.01394       0.06134  ...   
2           0.00009   0.00544   0.00781     0.01633       0.05233  ...   
3           0.00009   0.00502   0.00698     0.01505       0.05492  ...   
4           0.00011   0.00655   0.00908     0.01966       0.06425  ...   

   Shimmer:DDA      NHR     HNR  status      RPDE       DFA  

In [70]:
import re
import json
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib
import os

# Extract subject ID for Group-level splits
def extract_subject(name):
    match = re.search(r'S\d+', name)
    return match.group(0) if match else name

df['subject_id'] = df['name'].apply(extract_subject)

# Define features and targets
feature_cols = [col for col in df.columns if col not in ['name', 'status', 'subject_id']]
X = df[feature_cols]
y = df['status']
groups = df['subject_id']

print(f"Unique subjects: {df['subject_id'].nunique()}")
print(f"Target class balance:\n{df['status'].value_counts()}")

Unique subjects: 32
Target class balance:
status
1    147
0     48
Name: count, dtype: int64


In [71]:
sgkf = StratifiedGroupKFold(n_splits=5)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5))
])

fold_metrics = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:, 1]

    fold_metrics.append({
        'fold': fold + 1,
        'accuracy': accuracy_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'f1_score': f1_score(y_test, preds),
        'roc_auc': roc_auc_score(y_test, probs)
    })

metrics_df = pd.DataFrame(fold_metrics)
print("=== STRATIFIED GROUP K-FOLD CROSS-VALIDATION RESULTS ===")
print(metrics_df)
print("\nMean Metrics:")
print(metrics_df.mean())

=== STRATIFIED GROUP K-FOLD CROSS-VALIDATION RESULTS ===
   fold  accuracy    recall  precision  f1_score   roc_auc
0     1  0.918919  0.903226   1.000000  0.949153  0.989247
1     2  0.702703  1.000000   0.694444  0.819672  0.920000
2     3  0.945946  0.967742   0.967742  0.967742  0.962366
3     4  0.738095  0.833333   0.806452  0.819672  0.736111
4     5  0.666667  0.933333   0.700000  0.800000  0.766667

Mean Metrics:
fold         3.000000
accuracy     0.794466
recall       0.927527
precision    0.833728
f1_score     0.871248
roc_auc      0.874878
dtype: float64


In [ ]:
# Train final model pipeline and export matching the design
pipeline.fit(X, y)
output_dir = 'Multiple_disease_prediction_system/models/parkinsons'
os.makedirs(output_dir, exist_ok=True)

model_file = os.path.join(output_dir, 'parkinsons_model.pkl')
preprocessor_file = os.path.join(output_dir, 'preprocessor.pkl')

joblib.dump(pipeline.named_steps['classifier'], model_file)
joblib.dump(pipeline.named_steps['scaler'], preprocessor_file)

metadata = {
    "disease": "parkinsons",
    "features": feature_cols,
    "metrics": metrics_df.mean().to_dict()
}
with open(os.path.join(output_dir, 'parkinsons_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

print("Artifacts exported:", os.listdir(output_dir))

Artifacts exported: ['preprocessor.pkl', 'parkinsons_model.pkl', 'parkinsons_metadata.json']
